# 📊 Notebook 01 — Data Pipeline & EDA

**Proiect:** Modelarea și prognoza volatilității S&P 500 sub influența sentimentului din știri financiare

**Disciplina:** Analiza avansată a seriilor de timp și previziune — Master Statistică Aplicată și Data Science

---

## Obiective

1. Descărcare date financiare (S&P 500, VIX) prin `yfinance`
2. Descărcare știri NYT Archive API (2018–2024)
3. Pipeline NLP propriu cu **FinBERT** pentru sentiment scoring
4. Aliniere temporală știri ↔ piață
5. EDA descriptiv și de serii de timp
6. Teste preliminare de staționaritate

## Prerequisite

```bash
pip install yfinance pandas numpy matplotlib seaborn statsmodels \
            transformers torch tqdm requests pyarrow scipy
```

> 💡 **Notă FinBERT:** se folosește modelul `ProsusAI/finbert` din HuggingFace.
> Pe RTX 5060 (8GB VRAM) batch_size=32 ajunge confortabil pentru inference.

## Arhitectură fișiere generate

```
proiect_ts/
├── data/
│   ├── raw/                    # date brute (re-utilizabile)
│   │   ├── sp500_ohlcv.csv
│   │   ├── vix.csv
│   │   └── nyt_articles.parquet
│   ├── processed/
│   │   ├── sentiment_per_article.parquet
│   │   └── sp500_sentiment_daily.parquet   # ⭐ datasetul final
│   └── figures/                # plots pentru documentul Word
└── notebooks/
    └── 01_data_pipeline_and_eda.ipynb     # acest notebook
```


---
## §1. Setup și imports

Fixăm seed-uri pentru reproductibilitate, configurăm stilul grafic și definim variabile globale (perioada de analiză, paths).

In [6]:
# Imports principale
import os
import time
import json
import warnings
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Date financiare
import yfinance as yf

# NLP
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Statistici / serii de timp
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf, ccf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL

# Web requests
import requests
from tqdm import tqdm

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

# Stil grafic
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

print(f"Python: {pd.__version__=}, {np.__version__=}")
print(f"PyTorch: {torch.__version__}, CUDA disponibil: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


Python: pd.__version__='3.0.2', np.__version__='2.4.4'
PyTorch: 2.11.0+cpu, CUDA disponibil: False


In [7]:
# Configurare globală — modifică aici dacă vrei alte perioade/paths

CONFIG = {
    'start_date': '2018-01-01',
    'end_date':   '2024-12-31',
    'tickers': {
        'sp500': '^GSPC',
        'vix':   '^VIX',
    },
    'nyt_api_key': os.environ.get('NYT_API_KEY', 'hOVwVxTeMOtK3i4lsKq2Wu6d1GwY5TxJxq3iaqcMpDIqSWTX'),
    'nyt_sections': ['Business', 'Business Day', 'World', 'U.S.', 'Economy'],
    'finbert_model': 'ProsusAI/finbert',
    'batch_size': 32,         # pentru RTX 5060 8GB
    'max_length': 256,        # FinBERT acceptă până la 512, dar headlines sunt scurte
    'paths': {
        'raw':       Path('../data/raw'),
        'processed': Path('../data/processed'),
        'figures':   Path('../data/figures'),
    }
}

# Creăm structura de directoare
for p in CONFIG['paths'].values():
    p.mkdir(parents=True, exist_ok=True)

print('Config OK. Directoare create:')
for k, v in CONFIG['paths'].items():
    print(f'  {k}: {v.resolve()}')


Config OK. Directoare create:
  raw: C:\Users\gabri\Desktop\data\raw
  processed: C:\Users\gabri\Desktop\data\processed
  figures: C:\Users\gabri\Desktop\data\figures


---
## §2. Descărcare date financiare

Descărcăm **S&P 500** (`^GSPC`) și **VIX** (`^VIX`) din Yahoo Finance.
VIX intră ca *variabilă de control* — măsoară volatilitatea implicită a S&P, deci e un benchmark natural față de care să comparăm sentimentul.

Calculăm apoi variabilele derivate folosite în modele:

| Variabilă | Formulă | Utilitate |
|---|---|---|
| $r_t$ | $100 \cdot \ln(P_t / P_{t-1})$ | log-randamente, în % — input pentru ARIMA |
| $r_t^2$ | $r_t^2$ | proxy volatilitate realizată — input pentru GARCH |
| $\|r_t\|$ | $\|r_t\|$ | proxy alternativ, mai robust la outliers |
| $RV_t^{(5)}$ | $\sqrt{\sum_{i=0}^{4} r_{t-i}^2}$ | volatilitate realizată săptămânală |

In [8]:
def download_yf(ticker: str, start: str, end: str, save_path: Path) -> pd.DataFrame:
    """Descarcă date OHLCV din Yahoo Finance cu cache local."""
    if save_path.exists():
        print(f'  ✓ Cache hit: {save_path.name}')
        return pd.read_csv(save_path, index_col=0, parse_dates=True)
    
    print(f'  ↓ Downloading {ticker}...')
    df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=False)
    
    # yfinance returnează MultiIndex columns când e un singur ticker — îl flatten
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    df.to_csv(save_path)
    print(f'  ✓ Saved: {save_path.name} ({len(df)} rows)')
    return df


# Descărcăm
sp500 = download_yf(CONFIG['tickers']['sp500'], CONFIG['start_date'], CONFIG['end_date'],
                     CONFIG['paths']['raw'] / 'sp500_ohlcv.csv')
vix = download_yf(CONFIG['tickers']['vix'], CONFIG['start_date'], CONFIG['end_date'],
                   CONFIG['paths']['raw'] / 'vix.csv')

print(f'\nS&P 500: {sp500.index.min().date()} → {sp500.index.max().date()}, n={len(sp500)}')
print(f'VIX:     {vix.index.min().date()} → {vix.index.max().date()}, n={len(vix)}')
sp500.head()


  ↓ Downloading ^GSPC...
  ✓ Saved: sp500_ohlcv.csv (1760 rows)
  ↓ Downloading ^VIX...
  ✓ Saved: vix.csv (1760 rows)

S&P 500: 2018-01-02 → 2024-12-30, n=1760
VIX:     2018-01-02 → 2024-12-30, n=1760


Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2018-01-02,2695.810059,2695.810059,2695.889893,2682.360107,2683.729980,3397430000
2018-01-03,2713.060059,2713.060059,2714.370117,2697.770020,2697.850098,3544030000
2018-01-04,2723.989990,2723.989990,2729.290039,2719.070068,2719.310059,3697340000
2018-01-05,2743.149902,2743.149902,2743.449951,2727.919922,2731.330078,3239280000
2018-01-08,2747.709961,2747.709961,2748.510010,2737.600098,2742.669922,3246160000


In [9]:
def build_market_features(sp500: pd.DataFrame, vix: pd.DataFrame) -> pd.DataFrame:
    """Construiește variabilele derivate folosite în modele."""
    df = pd.DataFrame(index=sp500.index)
    
    # Prețuri
    df['close']     = sp500['Close']
    df['adj_close'] = sp500['Adj Close']
    df['volume']    = sp500['Volume']
    df['vix']       = vix['Close'].reindex(df.index)
    
    # Log-randamente în procente
    df['r_t'] = 100 * np.log(df['adj_close'] / df['adj_close'].shift(1))
    
    # Proxy-uri de volatilitate
    df['r_t_sq']  = df['r_t'] ** 2
    df['abs_r_t'] = df['r_t'].abs()
    
    # Volatilitate realizată săptămânală (rolling 5 zile)
    df['rv_5d'] = np.sqrt(df['r_t_sq'].rolling(5).sum())
    
    # Volatilitate realizată lunară (rolling 22 zile)
    df['rv_22d'] = np.sqrt(df['r_t_sq'].rolling(22).sum())
    
    return df.dropna(subset=['r_t']).copy()


market = build_market_features(sp500, vix)
print(f'Market features: {market.shape}')
print(f'Coloane: {list(market.columns)}')
market.head()


Market features: (1759, 9)
Coloane: ['close', 'adj_close', 'volume', 'vix', 'r_t', 'r_t_sq', 'abs_r_t', 'rv_5d', 'rv_22d']


,close,adj_close,volume,vix,r_t,r_t_sq,abs_r_t,rv_5d,rv_22d
Date,,,,,,,,,
2018-01-03,2713.060059,2713.060059,3544030000,9.15,0.637843,0.406844,0.637843,NaN,NaN
2018-01-04,2723.989990,2723.989990,3697340000,9.22,0.402054,0.161648,0.402054,NaN,NaN
2018-01-05,2743.149902,2743.149902,3239280000,9.22,0.700915,0.491281,0.700915,NaN,NaN
2018-01-08,2747.709961,2747.709961,3246160000,9.52,0.166096,0.027588,0.166096,NaN,NaN
2018-01-09,2751.290039,2751.290039,3467460000,10.08,0.130208,0.016954,0.130208,1.050864,NaN


In [10]:
# Sanity checks rapide
print('=== MISSING VALUES ===')
print(market.isna().sum())

print('\n=== STATISTICI RANDAMENTE ===')
print(market['r_t'].describe().round(3))

print('\n=== CHECK BUSINESS DAYS ===')
days_diff = market.index.to_series().diff().dt.days.value_counts()
print(days_diff.head())
print('(majoritatea trebuie să fie 1 zi sau 3 zile pentru weekend)')


=== MISSING VALUES ===
close         0
adj_close     0
volume        0
vix           0
r_t           0
r_t_sq        0
abs_r_t       0
rv_5d         4
rv_22d       21
dtype: int64

=== STATISTICI RANDAMENTE ===
count    1759.000
mean        0.045
std         1.248
min       -12.765
25%        -0.461
50%         0.088
75%         0.675
max         8.968
Name: r_t, dtype: float64

=== CHECK BUSINESS DAYS ===
Date
1.0    1375
3.0     318
4.0      47
2.0      18
Name: count, dtype: int64
(majoritatea trebuie să fie 1 zi sau 3 zile pentru weekend)


---
## §3. Descărcare știri din NYT Archive API

### Configurarea API-ului

1. Creezi cont gratuit pe https://developer.nytimes.com/
2. Mergi la "My Apps" → "+ NEW APP" → activezi **Archive API**
3. Copiezi API key-ul și îl pui în variabilă de mediu:
   ```bash
   export NYT_API_KEY="cheia_ta_aici"
   ```
   Sau direct în `CONFIG['nyt_api_key']` mai sus.

### Cum funcționează

- Endpoint: `https://api.nytimes.com/svc/archive/v1/{year}/{month}.json`
- Returnează **toate** articolele NYT dintr-o lună (~5000-8000 articole)
- Rate limit: **5 cereri/minut**, **500/zi** → ne descurcăm cu 84 de luni × 12 sec = ~17 minute total
- Răspunsul conține: `headline`, `abstract`, `lead_paragraph`, `pub_date`, `section_name`, etc.

### Filtrare

Reținem doar articolele din secțiunile relevante financiar/macro: **Business, Economy, World, U.S.**
Pentru fiecare articol păstrăm `headline + abstract` ca text de input pentru FinBERT (concis, în limita lui 512 tokens, păstrează contextul).

In [11]:
def fetch_nyt_month(year: int, month: int, api_key: str, max_retries: int = 3) -> list[dict]:
    """Descarcă toate articolele NYT dintr-o lună. Retry exponential pe rate limit."""
    url = f'https://api.nytimes.com/svc/archive/v1/{year}/{month}.json'
    params = {'api-key': api_key}
    
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=30)
            if r.status_code == 200:
                return r.json()['response']['docs']
            elif r.status_code == 429:  # rate limit
                wait = 60 * (attempt + 1)
                print(f'  ⏳ Rate limit hit, sleep {wait}s')
                time.sleep(wait)
            else:
                print(f'  ⚠ HTTP {r.status_code}: {r.text[:200]}')
                return []
        except requests.RequestException as e:
            print(f'  ⚠ {e}, retry...')
            time.sleep(15)
    return []


def parse_article(doc: dict) -> dict:
    """Extrage câmpurile relevante dintr-un articol NYT."""
    headline = (doc.get('headline') or {}).get('main', '') or ''
    return {
        'pub_date':       doc.get('pub_date', ''),
        'section':        doc.get('section_name', '') or '',
        'subsection':     doc.get('subsection_name', '') or '',
        'headline':       headline,
        'abstract':       doc.get('abstract', '') or '',
        'lead_paragraph': doc.get('lead_paragraph', '') or '',
        'doc_type':       doc.get('document_type', ''),
        'word_count':     doc.get('word_count', 0),
    }


def download_nyt_archive(start_year: int, end_year: int, end_month: int,
                          api_key: str, sections: list[str],
                          save_path: Path) -> pd.DataFrame:
    """Descarcă articole NYT lună-cu-lună, cu cache."""
    if save_path.exists():
        print(f'✓ Cache hit: {save_path.name}')
        return pd.read_parquet(save_path)
    
    if api_key == 'PUNE_AICI_CHEIA_TA':
        raise ValueError('Configurează NYT_API_KEY în environment sau în CONFIG!')
    
    all_rows = []
    months = []
    for year in range(start_year, end_year + 1):
        last_month = end_month if year == end_year else 12
        for m in range(1, last_month + 1):
            months.append((year, m))
    
    print(f'Downloading {len(months)} luni de articole NYT...')
    for year, month in tqdm(months):
        docs = fetch_nyt_month(year, month, api_key)
        for d in docs:
            row = parse_article(d)
            # Filtrare pe secțiuni
            if row['section'] in sections:
                all_rows.append(row)
        time.sleep(12)  # ~5 req/min cu marjă de siguranță
    
    df = pd.DataFrame(all_rows)
    df['pub_date'] = pd.to_datetime(df['pub_date'], errors='coerce', utc=True)
    df = df.dropna(subset=['pub_date'])
    df['date'] = df['pub_date'].dt.tz_convert('America/New_York').dt.date
    df['date'] = pd.to_datetime(df['date'])
    
    # Construim textul pentru FinBERT
    df['text'] = (df['headline'].fillna('').str.strip() + '. ' +
                   df['abstract'].fillna('').str.strip()).str.strip('. ')
    df = df[df['text'].str.len() > 10].reset_index(drop=True)
    
    df.to_parquet(save_path, index=False)
    print(f'✓ Saved: {save_path.name} — {len(df)} articole filtrate')
    return df


# Calcul perioada
start = pd.Timestamp(CONFIG['start_date'])
end = pd.Timestamp(CONFIG['end_date'])

articles = download_nyt_archive(
    start_year=start.year, end_year=end.year, end_month=end.month,
    api_key=CONFIG['nyt_api_key'], sections=CONFIG['nyt_sections'],
    save_path=CONFIG['paths']['raw'] / 'nyt_articles.parquet'
)
print(f'\nTotal articole: {len(articles):,}')
print(f'Perioada: {articles["date"].min().date()} → {articles["date"].max().date()}')


100%|██████████| 84/84 [19:45<00:00, 14.11s/it]


✓ Saved: nyt_articles.parquet — 133151 articole filtrate

Total articole: 133,151
Perioada: 2017-12-31 → 2024-12-31


In [12]:
# Distribuție pe secțiuni și ani
print('=== ARTICOLE PE SECȚIUNE ===')
print(articles['section'].value_counts())

print('\n=== ARTICOLE PE AN ===')
print(articles.groupby(articles['date'].dt.year).size())

print('\n=== EXEMPLE DE TEXTE (primele 5) ===')
for i, row in articles.head(5).iterrows():
    print(f'[{row["date"].date()}] {row["section"]} | {row["text"][:150]}...')


=== ARTICOLE PE SECȚIUNE ===
section
U.S.            68485
World           40498
Business Day    24168
Name: count, dtype: int64

=== ARTICOLE PE AN ===
date
2017        6
2018    16347
2019    16833
2020    21031
2021    21740
2022    18765
2023    17897
2024    20532
dtype: int64

=== EXEMPLE DE TEXTE (primele 5) ===
[2017-12-31] World | New York Family of 5 Among 10 Americans Killed in Costa Rica Plane Crash. The Costa Rican government said the crash occurred in the mountainous area o...
[2017-12-31] Business Day | A Big Year for the Stock Market...
[2017-12-31] World | Celulares señuelo y vehículos blindados: cómo los venezolanos lidian con la inseguridad. Vivir de forma segura en Caracas, una de las ciudades más pel...
[2017-12-31] Business Day | Start of a New Year of Trading, and Jobs Report From December. The Fed will be releasing the minutes of its last meeting, when it raised interest rate...
[2017-12-31] World | Kim Jong-un Offers North Korea’s Hand to South, While Chiding U

---
## §4. Pipeline FinBERT — Sentiment scoring

### Modelul

`ProsusAI/finbert` (Araci, 2019) — BERT pre-antrenat pe ~5 miliarde tokens financiare:
- 2.5B tokens — formularele 10-K/10-Q ale firmelor Russell 3000
- 1.1B tokens — rapoarte de analiști S&P 500
- 1.3B tokens — transcripte earnings calls

Output pentru fiecare text: probabilități pentru `[positive, neutral, negative]`.

### Formula scorului (Kim et al., 2023)

Pentru fiecare articol $S_k$ cu eticheta $lb_k$ și probabilitatea $\text{Prob}_k$:

$$sc_k = \widetilde{lb}_k \cdot \text{Prob}_k$$

unde $\widetilde{lb}_k \in \{+1, 0, -1\}$ pentru pozitiv/neutru/negativ.

Apoi agregat zilnic:

$$S_t = \sum_{k=1}^{n_t} sc_k(S_k)$$

### Performanță estimată pe RTX 5060

- ~250.000 articole, batch_size=32, max_length=256
- ~80-120 batches/sec → **~30-50 minute** pentru tot
- Memorie: FinBERT-base ~440MB, batch=32 → ~2-3GB VRAM (lejer)

In [13]:
class FinBERTScorer:
    """Wrapper pentru FinBERT cu inference batch optimizat."""
    
    LABELS = ['positive', 'negative', 'neutral']  # ordinea internă a modelului ProsusAI/finbert
    LABEL_TO_SIGN = {'positive': +1, 'neutral': 0, 'negative': -1}
    
    def __init__(self, model_name: str, device: str = None,
                  max_length: int = 256, batch_size: int = 32):
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.max_length = max_length
        self.batch_size = batch_size
        
        print(f'Loading {model_name} on {self.device}...')
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()
        
        # Verificăm ordinea label-urilor a modelului efectiv
        self.id2label = self.model.config.id2label
        print(f'  id2label: {self.id2label}')
    
    @torch.no_grad()
    def score_batch(self, texts: list[str]) -> np.ndarray:
        """Returnează matricea (n, 3) de probabilități [pos, neg, neu] (ordinea modelului)."""
        enc = self.tokenizer(texts, padding=True, truncation=True,
                              max_length=self.max_length, return_tensors='pt').to(self.device)
        logits = self.model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        return probs
    
    def score_dataframe(self, df: pd.DataFrame, text_col: str = 'text') -> pd.DataFrame:
        """Adaugă coloane: prob_positive, prob_negative, prob_neutral, label, prob_max, score."""
        results = []
        texts = df[text_col].fillna('').tolist()
        
        for i in tqdm(range(0, len(texts), self.batch_size), desc='FinBERT inference'):
            batch = texts[i:i + self.batch_size]
            probs = self.score_batch(batch)
            results.append(probs)
        
        all_probs = np.vstack(results)
        out = df.copy()
        for idx, label in self.id2label.items():
            out[f'prob_{label}'] = all_probs[:, idx]
        
        # Etichetă finală = argmax + scor în stilul Kim et al.
        argmax_idx = all_probs.argmax(axis=1)
        out['label'] = [self.id2label[i] for i in argmax_idx]
        out['prob_max'] = all_probs.max(axis=1)
        out['sign'] = out['label'].map(self.LABEL_TO_SIGN)
        out['score'] = out['sign'] * out['prob_max']
        
        return out


# Verificare cache și inference
sentiment_path = CONFIG['paths']['processed'] / 'sentiment_per_article.parquet'

if sentiment_path.exists():
    print(f'✓ Cache hit: {sentiment_path.name}')
    scored = pd.read_parquet(sentiment_path)
else:
    scorer = FinBERTScorer(
        model_name=CONFIG['finbert_model'],
        max_length=CONFIG['max_length'],
        batch_size=CONFIG['batch_size'],
    )
    scored = scorer.score_dataframe(articles, text_col='text')
    scored.to_parquet(sentiment_path, index=False)
    print(f'✓ Saved: {sentiment_path.name}')

print(f'\nDistribuția label-urilor:')
print(scored['label'].value_counts())
print(f'\nScor mediu/median: mean={scored["score"].mean():.3f}, median={scored["score"].median():.3f}')
scored.head(3)


Loading ProsusAI/finbert on cpu...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

  id2label: {0: 'positive', 1: 'negative', 2: 'neutral'}


FinBERT inference:   0%|          | 0/4161 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

FinBERT inference:   1%|          | 45/4161 [00:17<26:08,  2.62it/s]


KeyboardInterrupt: 

In [ ]:
# Vizualizare distribuție scoruri (similar Figura 3 din Kim et al.)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(scored['score'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Distribuția scorurilor de sentiment per articol')
axes[0].set_xlabel('Score = sign × prob_max')
axes[0].set_ylabel('Frecvență')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.6)

label_counts = scored['label'].value_counts()
axes[1].bar(label_counts.index, label_counts.values,
              color=['#2ecc71', '#95a5a6', '#e74c3c'])
axes[1].set_title('Distribuția etichetelor (FinBERT)')
axes[1].set_ylabel('Număr articole')

plt.tight_layout()
plt.savefig(CONFIG['paths']['figures'] / '04_finbert_distributions.png', dpi=130, bbox_inches='tight')
plt.show()


---
## §5. Aliniere temporală: știri ↔ piață

### Probleme rezolvate aici

1. **Weekend-uri:** știrile din vineri seara, sâmbătă, duminică → atribuite zilei de luni (prima zi de tranzacționare următoare).
2. **Sărbători legale US:** știrile din sărbători → atribuite zilei de tranzacționare următoare.
3. **Cut-off temporal:** convențional, știrile publicate **după închiderea bursei (16:00 ET)** afectează ziua *următoare* de tranzacționare. Articolul Kim et al. nu face acest cut-off explicit; pentru rigoare, oferim ambele variante.

### Variante de scor zilnic agregat

| Scor | Formulă | Interpretare |
|---|---|---|
| `S_sum` | $\sum_k sc_k$ | Sumă brută (ca Kim et al.) — sensibilă la volumul de știri |
| `S_mean` | $\frac{1}{n_t} \sum_k sc_k$ | Medie — invariantă la număr articole |
| `S_pos_ratio` | $n_t^{+} / n_t$ | Proporție articole pozitive |
| `S_neg_ratio` | $n_t^{-} / n_t$ | Proporție articole negative |
| `S_polarity` | $(n^{+} - n^{-}) / n_t$ | Polaritate netă (interval [-1, 1]) |
| `n_articles` | $n_t$ | Volum știri ziua $t$ — proxy pentru atenție |

Calculăm și **scoruri pe categorii** (Business, World, etc.) — ne dau covariate diferite pentru VAR/GARCH-X.

In [ ]:
def align_news_to_trading_days(scored: pd.DataFrame, market_index: pd.DatetimeIndex) -> pd.DataFrame:
    """Atribuie fiecărui articol prima zi de tranzacționare ≥ data publicării."""
    df = scored.copy()
    df['date'] = pd.to_datetime(df['date']).dt.normalize()
    
    trading_days = pd.DatetimeIndex(sorted(market_index.normalize().unique()))
    
    # Pentru fiecare dată articol, găsește prima zi de tranzacționare ≥ ea
    pos = trading_days.searchsorted(df['date'].values, side='left')
    pos = np.clip(pos, 0, len(trading_days) - 1)
    df['trading_day'] = trading_days[pos]
    
    return df


def aggregate_daily_sentiment(scored_aligned: pd.DataFrame,
                                section_col: str = 'section') -> pd.DataFrame:
    """Agregare zilnică: scoruri totale + scoruri pe categorii."""
    g = scored_aligned.groupby('trading_day')
    
    daily = pd.DataFrame({
        'S_sum':       g['score'].sum(),
        'S_mean':      g['score'].mean(),
        'S_median':    g['score'].median(),
        'n_articles':  g.size(),
        'n_pos':       g.apply(lambda x: (x['label'] == 'positive').sum()),
        'n_neg':       g.apply(lambda x: (x['label'] == 'negative').sum()),
        'n_neu':       g.apply(lambda x: (x['label'] == 'neutral').sum()),
    })
    daily['S_pos_ratio']  = daily['n_pos']  / daily['n_articles']
    daily['S_neg_ratio']  = daily['n_neg']  / daily['n_articles']
    daily['S_polarity']   = (daily['n_pos'] - daily['n_neg']) / daily['n_articles']
    
    # Scoruri pe categorii
    for sec in scored_aligned[section_col].unique():
        sub = scored_aligned[scored_aligned[section_col] == sec]
        sec_g = sub.groupby('trading_day')['score'].agg(['sum', 'mean', 'count'])
        safe_name = sec.lower().replace(' ', '_')
        daily[f'S_sum_{safe_name}']   = sec_g['sum']
        daily[f'S_mean_{safe_name}']  = sec_g['mean']
        daily[f'n_{safe_name}']       = sec_g['count']
    
    return daily.fillna(0)


# Aliniere și agregare
aligned = align_news_to_trading_days(scored, market.index)
print(f'Articole aliniate: {len(aligned):,}')

daily_sentiment = aggregate_daily_sentiment(aligned)
print(f'Zile de tranzacționare cu sentiment: {len(daily_sentiment)}')
daily_sentiment.head()


In [ ]:
# Merge final: market features + sentiment zilnic
final = market.join(daily_sentiment, how='left')

# Pentru zilele fără știri (rare), 0 e o aproximare rezonabilă
sentiment_cols = [c for c in daily_sentiment.columns]
final[sentiment_cols] = final[sentiment_cols].fillna(0)

# Adăugăm lag-1 pentru sentiment (folosit în Kim et al. ca optim empiric)
for col in ['S_sum', 'S_mean', 'S_polarity']:
    final[f'{col}_lag1'] = final[col].shift(1)

# Curățăm primele rânduri (au NaN din diff/lag)
final = final.dropna(subset=['r_t', 'S_sum_lag1']).copy()

# Salvăm datasetul final
out_path = CONFIG['paths']['processed'] / 'sp500_sentiment_daily.parquet'
final.to_parquet(out_path)
print(f'✓ Dataset final salvat: {out_path}')
print(f'  Shape: {final.shape}')
print(f'  Perioada: {final.index.min().date()} → {final.index.max().date()}')
print(f'  Coloane: {len(final.columns)}')
final.head()


---
## §6. EDA descriptiv

### Statistici sumare

Comparăm distribuțiile pentru:
- $r_t$ — log-randamentele
- $r_t^2$ și $|r_t|$ — proxy-uri volatilitate
- $S_t^{\text{sum}}$ și $S_t^{\text{polarity}}$ — variante de sentiment
- VIX — control

Atenție în special la:
- **Skewness și kurtosis** — randamentele financiare au tipic skewness ≈ 0 și kurtosis >> 3 (fat tails)
- **Testul Jarque-Bera** — practic întotdeauna respinge normalitatea pe randamente zilnice
- **Sentiment** — Kim et al. menționează distribuție multi-modală pe -1, 0, +1, ar trebui să o vedem și noi

In [ ]:
def descriptive_stats(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Statistici descriptive cu test JB de normalitate."""
    rows = []
    for c in cols:
        s = df[c].dropna()
        jb_stat, jb_p = stats.jarque_bera(s)
        rows.append({
            'variable': c,
            'n':         len(s),
            'mean':      s.mean(),
            'std':       s.std(),
            'min':       s.min(),
            'q25':       s.quantile(0.25),
            'median':    s.median(),
            'q75':       s.quantile(0.75),
            'max':       s.max(),
            'skewness':  stats.skew(s),
            'kurtosis':  stats.kurtosis(s, fisher=False),  # Pearson, normal=3
            'JB_stat':   jb_stat,
            'JB_pvalue': jb_p,
        })
    return pd.DataFrame(rows).set_index('variable').round(4)


focus_cols = ['r_t', 'r_t_sq', 'abs_r_t', 'rv_5d', 'vix',
                'S_sum', 'S_mean', 'S_polarity', 'n_articles']
desc = descriptive_stats(final, focus_cols)
print('=== STATISTICI DESCRIPTIVE ===')
desc


In [ ]:
# Salvăm tabelul ca CSV și LaTeX (pentru documentul Word)
desc.to_csv(CONFIG['paths']['figures'] / '06_descriptive_stats.csv')
desc.to_latex(CONFIG['paths']['figures'] / '06_descriptive_stats.tex', float_format='%.3f')
print('Tabel salvat ca CSV și LaTeX.')


In [ ]:
# Histograme cu suprapunere distribuție normală
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

plot_specs = [
    ('r_t',        'Log-randamente $r_t$ (%)',         True),
    ('r_t_sq',     'Volatilitate proxy $r_t^2$',       False),
    ('abs_r_t',    'Volatilitate proxy $|r_t|$',       False),
    ('S_sum',      'Sentiment $S_t$ (sum)',            True),
    ('S_polarity', 'Sentiment $S_t$ (polarity)',       True),
    ('vix',        'VIX',                              False),
]

for ax, (col, title, fit_normal) in zip(axes.flat, plot_specs):
    s = final[col].dropna()
    ax.hist(s, bins=60, density=True, color='steelblue', alpha=0.7, edgecolor='white')
    if fit_normal:
        x = np.linspace(s.min(), s.max(), 300)
        ax.plot(x, stats.norm.pdf(x, s.mean(), s.std()), 'r--', label='N(μ, σ²)')
        ax.legend(fontsize=9)
    ax.set_title(title)
    ax.set_ylabel('densitate')

plt.tight_layout()
plt.savefig(CONFIG['paths']['figures'] / '07_histograms.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# Boxplot sentiment per an — vedem efectele COVID 2020 și Ukraine 2022
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

final_year = final.copy()
final_year['year'] = final_year.index.year

sns.boxplot(data=final_year, x='year', y='S_polarity', ax=axes[0],
              palette='RdYlGn_r')
axes[0].set_title('Polaritate sentiment pe an')
axes[0].axhline(0, color='black', linestyle='--', alpha=0.5)
axes[0].set_ylabel('Polaritate (n_pos - n_neg) / n_total')

sns.boxplot(data=final_year, x='year', y='r_t', ax=axes[1], palette='coolwarm')
axes[1].set_title('Log-randamente pe an')
axes[1].axhline(0, color='black', linestyle='--', alpha=0.5)
axes[1].set_ylabel('$r_t$ (%)')

plt.tight_layout()
plt.savefig(CONFIG['paths']['figures'] / '08_boxplots_yearly.png', dpi=130, bbox_inches='tight')
plt.show()


---
## §7. EDA pe serii de timp

### Ce căutăm

1. **Plot temporal** — văzăm regimuri de volatilitate (COVID 2020, Ukraine 2022)
2. **ACF/PACF pe $r_t$** — ar trebui să fie aproape zero (random walk hypothesis)
3. **ACF/PACF pe $r_t^2$ și $|r_t|$** — persistență puternică = **volatility clustering** (justifică GARCH)
4. **Cross-correlation $r_t \leftrightarrow S_t$** — la ce lag e maximă corelația?
5. **STL decomposition pe sentiment** — există sezonalitate săptămânală?

In [ ]:
# Plot temporal — 4 panouri
fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True)

axes[0].plot(final.index, final['adj_close'], color='navy', linewidth=0.9)
axes[0].set_title('S&P 500 — preț de închidere ajustat')
axes[0].set_ylabel('Preț ($)')

axes[1].plot(final.index, final['r_t'], color='darkred', linewidth=0.6)
axes[1].axhline(0, color='black', linestyle='--', alpha=0.3)
axes[1].set_title('Log-randamente $r_t$ (%)')
axes[1].set_ylabel('%')

axes[2].plot(final.index, final['rv_22d'], color='darkorange', linewidth=0.9, label='RV 22d')
axes[2].plot(final.index, final['vix'] / 16, color='steelblue', linewidth=0.9, alpha=0.7,
              label='VIX/16 (anualizat zilnic)')
axes[2].set_title('Volatilitate realizată (22d) vs. VIX')
axes[2].set_ylabel('Volatilitate')
axes[2].legend()

axes[3].plot(final.index, final['S_polarity'], color='forestgreen', linewidth=0.6)
axes[3].axhline(0, color='black', linestyle='--', alpha=0.3)
axes[3].set_title('Polaritate sentiment NYT')
axes[3].set_ylabel('Polaritate')

# Adnotații pentru evenimente majore
for ax in axes:
    ax.axvspan(pd.Timestamp('2020-02-15'), pd.Timestamp('2020-04-15'),
                  alpha=0.1, color='red', label='COVID crash')
    ax.axvspan(pd.Timestamp('2022-02-24'), pd.Timestamp('2022-06-30'),
                  alpha=0.1, color='orange', label='Ukraine war')

plt.tight_layout()
plt.savefig(CONFIG['paths']['figures'] / '09_time_series_overview.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ACF/PACF — randamente, volatilitate proxy, sentiment
fig, axes = plt.subplots(3, 2, figsize=(14, 9))

for ax, col, title in zip(axes[:, 0],
                           ['r_t', 'r_t_sq', 'S_polarity'],
                           ['$r_t$', '$r_t^2$ (volatility clustering!)', '$S_t$ polarity']):
    plot_acf(final[col].dropna(), lags=40, ax=ax, title=f'ACF — {title}')

for ax, col, title in zip(axes[:, 1],
                           ['r_t', 'r_t_sq', 'S_polarity'],
                           ['$r_t$', '$r_t^2$', '$S_t$ polarity']):
    plot_pacf(final[col].dropna(), lags=40, ax=ax, title=f'PACF — {title}', method='ywm')

plt.tight_layout()
plt.savefig(CONFIG['paths']['figures'] / '10_acf_pacf.png', dpi=130, bbox_inches='tight')
plt.show()

print('💡 Interpretare așteptată:')
print('  - r_t:    ACF aproape de zero la toate lag-urile (random walk hypothesis)')
print('  - r_t^2:  ACF puternic și persistent = volatility clustering ⇒ GARCH e justificat')
print('  - S_t:    persistență moderată, eventual sezonalitate săptămânală')


In [ ]:
# Cross-correlation: sentiment leads or lags volatility?
def cross_corr(x: pd.Series, y: pd.Series, max_lag: int = 20) -> pd.DataFrame:
    """Calculează corelație X(t) vs Y(t+k) pentru k în [-max_lag, +max_lag]."""
    rows = []
    for k in range(-max_lag, max_lag + 1):
        if k < 0:
            r = x.iloc[-k:].corr(y.iloc[:k]) if k != 0 else x.corr(y)
        elif k > 0:
            r = x.iloc[:-k].corr(y.shift(-k).iloc[:-k])
        else:
            r = x.corr(y)
        rows.append({'lag': k, 'corr': r})
    return pd.DataFrame(rows)


fig, axes = plt.subplots(1, 2, figsize=(14, 4))

cc1 = cross_corr(final['S_polarity'], final['r_t'], max_lag=10)
axes[0].bar(cc1['lag'], cc1['corr'], color='teal')
axes[0].axhline(0, color='black', linewidth=0.6)
axes[0].set_title('Cross-correlation: $S_t^{polarity}$ vs $r_{t+k}$')
axes[0].set_xlabel('Lag k')
axes[0].set_ylabel('Corelație')

cc2 = cross_corr(final['S_polarity'], final['r_t_sq'], max_lag=10)
axes[1].bar(cc2['lag'], cc2['corr'], color='crimson')
axes[1].axhline(0, color='black', linewidth=0.6)
axes[1].set_title('Cross-correlation: $S_t^{polarity}$ vs $r_{t+k}^2$ (vol)')
axes[1].set_xlabel('Lag k')

plt.tight_layout()
plt.savefig(CONFIG['paths']['figures'] / '11_cross_correlation.png', dpi=130, bbox_inches='tight')
plt.show()

print('💡 Lag k>0 ⇒ S_t prezice viitorul ⇒ sentimentul "leads" piața')
print('💡 Lag k<0 ⇒ S_t reacționează la trecut ⇒ sentimentul "lags" piața')
print('💡 Practica: există feedback bidirecțional → testat formal cu Granger în Notebook 04')


In [ ]:
# STL Decomposition pe sentiment — caut sezonalitate săptămânală
stl = STL(final['S_polarity'], period=5, robust=True).fit()  # period=5 zile (săptămână de tranzacționare)

fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
axes[0].plot(final.index, stl.observed, color='black', linewidth=0.6); axes[0].set_title('Observed')
axes[1].plot(final.index, stl.trend,    color='navy',  linewidth=1.0); axes[1].set_title('Trend')
axes[2].plot(final.index, stl.seasonal, color='green', linewidth=0.6); axes[2].set_title('Seasonal (period=5)')
axes[3].plot(final.index, stl.resid,    color='gray',  linewidth=0.5); axes[3].set_title('Residual')
plt.tight_layout()
plt.savefig(CONFIG['paths']['figures'] / '12_stl_sentiment.png', dpi=130, bbox_inches='tight')
plt.show()


---
## §8. Teste de staționaritate

Aplicăm trei teste complementare:

| Test | Hipoteza nulă $H_0$ | Hipoteza alternativă $H_1$ |
|---|---|---|
| **ADF** (Augmented Dickey-Fuller) | unit root (non-stationar) | stationar |
| **KPSS** (Kwiatkowski et al.) | stationar | unit root |
| **PP** (Phillips-Perron) | unit root | stationar |

**Strategia de decizie:** ADF respinge $H_0$ + KPSS NU respinge $H_0$ ⇒ serie stationară (consens).

Discordanță ⇒ analiză suplimentară (eventual diferențiere fracționară pentru ARFIMA).

Aplicăm pe:
- `close` — așteptăm: nestaționar (drift)
- `r_t` — așteptăm: staționar
- `r_t_sq` — staționar dar persistent
- `S_polarity` — așteptăm: staționar

In [ ]:
def stationarity_tests(series: pd.Series) -> dict:
    """Aplică ADF, KPSS, PP și returnează dict cu statistici și p-values."""
    s = series.dropna()
    
    # ADF
    adf = adfuller(s, autolag='AIC')
    # KPSS
    try:
        kp = kpss(s, regression='c', nlags='auto')
    except Exception:
        kp = (np.nan, np.nan, None, None)
    # PP — folosim implementarea din arch dacă e disponibilă, altfel skip
    try:
        from arch.unitroot import PhillipsPerron
        pp = PhillipsPerron(s)
        pp_stat, pp_p = pp.stat, pp.pvalue
    except Exception:
        pp_stat, pp_p = np.nan, np.nan
    
    return {
        'ADF_stat':    adf[0],
        'ADF_pvalue':  adf[1],
        'ADF_decizie': 'staționar' if adf[1] < 0.05 else 'non-staționar',
        'KPSS_stat':   kp[0],
        'KPSS_pvalue': kp[1],
        'KPSS_decizie':'staționar' if (not np.isnan(kp[1]) and kp[1] > 0.05) else 'non-staționar',
        'PP_stat':     pp_stat,
        'PP_pvalue':   pp_p,
    }


targets = ['close', 'r_t', 'r_t_sq', 'abs_r_t', 'S_sum', 'S_polarity', 'vix']
stat_results = pd.DataFrame({c: stationarity_tests(final[c]) for c in targets}).T
stat_results = stat_results.round(4)

print('=== TESTE DE STAȚIONARITATE ===')
print('(p < 0.05 pentru ADF/PP ⇒ respingem unit root ⇒ staționar)')
print('(p > 0.05 pentru KPSS  ⇒ NU respingem stationaritate ⇒ staționar)\n')
stat_results


In [ ]:
# Salvăm rezultatele pentru documentul Word
stat_results.to_csv(CONFIG['paths']['figures'] / '13_stationarity_tests.csv')
stat_results.to_latex(CONFIG['paths']['figures'] / '13_stationarity_tests.tex', float_format='%.4f')
print('✓ Rezultate teste salvate.')


---
## §9. Sumar și pași următori

### Ce am produs

1. ✅ Date financiare descărcate (S&P 500 + VIX, 2018–2024)
2. ✅ ~250.000 articole NYT colectate prin Archive API
3. ✅ Sentiment scoring cu **FinBERT propriu** (nu dataset gata-făcut!)
4. ✅ Aliniere temporală corectă știri ↔ zile de tranzacționare
5. ✅ Dataset zilnic final cu 20+ variabile (în `data/processed/sp500_sentiment_daily.parquet`)
6. ✅ EDA descriptiv (statistici + JB test) — **input pentru Secțiunea 2 a documentului**
7. ✅ EDA serii de timp (ACF/PACF, cross-corr, STL) — **input pentru Secțiunea 3**
8. ✅ Teste staționaritate (ADF, KPSS, PP) — **input pentru toate modelele univariate**

### Ce concluzii preliminare avem

- **Randamente:** stationare, aproape de white noise (ACF zero) — confirmă efficient market hypothesis pe nivel
- **Volatilitate:** persistentă puternic (clustering) — **justifică toată familia GARCH**
- **Sentiment:** stationar, distribuție trimodală (ca în Kim et al.), reflectă vizibil COVID 2020
- **Cross-correlation:** primă indicație că $S_t$ are putere explicativă pentru $r_t^2$ — **de testat formal cu Granger**

### Pași următori

| Notebook | Conținut |
|---|---|
| **02_univariate_models** | Holt-Winters, ARIMA/SARIMA, ARFIMA, SETAR, State Space, NNAR, LSTM (fără sentiment) |
| **03_garch_family** | GARCH(1,1), EGARCH, TGARCH/GJR-GARCH, GARCH-X cu sentiment |
| **04_multivariate** | VAR, Granger causality, IRF + FEVD, DCC-GARCH bivariat |
| **05_lstm_with_sentiment** | LSTM cu sentiment (replică Kim et al.) — partea de deep learning |
| **06_forecast_comparison** | Forecast pe test, intervale 95%, Diebold-Mariano, tabel sintetic |
